In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from tqdm.auto import tqdm
from transformers import AutoModel
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


/home/thomas/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from model.pairwise_model_v3.data import (
    preprocess_dataframe,
    tokenize_dataframe_val
)

from model.pairwise_model_v3.model import PairwisePreferenceModel

In [8]:
# =====================================
# Configuration
# =====================================

TRAIN_PATH = "../data/train.csv"
MODEL_NAME = "microsoft/deberta-v3-small"

MAX_LENGTH = 512
BATCH_SIZE = 2

CHECKPOINT_DIR = "../models/pairwise_model_v3"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TEST_SIZE = 0.1
IS_FULL_DATASET = True
DATASET_LIMIT = 1000
RANDOM_STATE = 42

In [9]:
df = pd.read_csv(TRAIN_PATH)

train_df, val_df = preprocess_dataframe(
    df=df,
    test_size=TEST_SIZE,
    is_full_dataset=IS_FULL_DATASET,
    dataset_limit=DATASET_LIMIT,
    random_state=RANDOM_STATE
)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

backbone = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

pairwise_model = PairwisePreferenceModel(backbone).to(device)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2340.43it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

Using device: cuda


In [11]:
val_loader = tokenize_dataframe_val(
    val_df,
    tokenizer,
    MAX_LENGTH,
    BATCH_SIZE
)

Tokenizing: 100%|██████████| 5748/5748 [00:08<00:00, 651.11it/s] 


In [12]:
def evaluate_checkpoint(checkpoint_path):

    # -----------------------------
    # Load checkpoint
    # -----------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )
    
    pairwise_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    pairwise_model.eval()

    # -----------------------------
    # Validation
    # -----------------------------

    total_loss = 0.0
    num_examples = 0
    correct = 0

    all_probabilities = []

    with torch.no_grad():
        progress_bar = tqdm( val_loader, desc="Validation" )

        for enc_a, enc_b, labels in progress_bar:

            enc_a = {
                k: v.to(device)
                for k, v in enc_a.items()
            }

            enc_b = {
                k: v.to(device)
                for k, v in enc_b.items()
            }

            labels = labels.to(device)

            logits = pairwise_model(
                input_ids_a=enc_a["input_ids"],
                attention_mask_a=enc_a["attention_mask"],
                input_ids_b=enc_b["input_ids"],
                attention_mask_b=enc_b["attention_mask"]
            )

            loss = F.cross_entropy(
                logits,
                labels
            )

            probabilities = F.softmax(
                logits,
                dim=1
            )

            predictions = probabilities.argmax(
                dim=1
            )

            all_probabilities.append(
                probabilities.cpu()
            )

            # Probability assigned to correct class
            true_probabilities = probabilities[
                torch.arange(
                    labels.size(0),
                    device=labels.device
                ),
                labels
            ]

            # Log loss
            loss = -torch.log(
                true_probabilities
            ).mean()

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            num_examples += batch_size

            # Predictions
            predictions = probabilities.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

    # -----------------------------
    # Final metrics
    # -----------------------------

    all_probabilities = torch.cat(
        all_probabilities,
        dim=0
    )

    val_loss = total_loss / num_examples
    val_accuracy = correct / num_examples

    return {
        "epoch": checkpoint["epoch"],
        "train_loss": checkpoint["train_loss"],
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "probabilities": all_probabilities
    }


results = []
for i in range(5):
    result = evaluate_checkpoint(f"{CHECKPOINT_DIR}/epoch_{i + 1}.pt")
    results.append(result)

for result in results:
    print(
        f"Epoch {result['epoch']}: "
        f"Train Loss = {result['train_loss']:.4f}, "
        f"Val Loss = {result['val_loss']:.4f}, "
        f"Val Accuracy = {result['val_accuracy']:.4f}, "
    )

Validation: 100%|██████████| 2874/2874 [02:23<00:00, 20.06it/s]

Epoch 1: Train Loss = 1.0418, Val Loss = 1.0158, Val Accuracy = 0.4840, 
Epoch 2: Train Loss = 0.9888, Val Loss = 1.0090, Val Accuracy = 0.4955, 
Epoch 3: Train Loss = 0.8481, Val Loss = 1.0789, Val Accuracy = 0.4770, 
Epoch 4: Train Loss = 0.6408, Val Loss = 1.3267, Val Accuracy = 0.4581, 
Epoch 5: Train Loss = 0.4647, Val Loss = 1.7226, Val Accuracy = 0.4457, 


In [16]:
for result in results:
    print(
        f"Epoch {result['epoch']}: "
        f"Train Loss = {result['train_loss']:.4f}, "
        f"Val Loss = {result['val_loss']:.4f}, "
        f"Val Accuracy = {result['val_accuracy']:.4f}, "
    )

Epoch 1: Train Loss = 1.1139, Val Loss = 1.0945, Val Accuracy = 0.4300, 
Epoch 2: Train Loss = 1.0431, Val Loss = 1.0561, Val Accuracy = 0.3900, 
Epoch 3: Train Loss = 0.9457, Val Loss = 1.1393, Val Accuracy = 0.3100, 
Epoch 4: Train Loss = 0.7351, Val Loss = 1.2191, Val Accuracy = 0.3800, 


In [17]:
# for result in results:
#     print(
#         f"Epoch {result['epoch']}: "
#         f"Train Loss = {result['train_loss']:.4f}, "
#         f"Val Loss = {result['val_loss']:.4f}, "
#         f"Val Accuracy = {result['val_accuracy']:.4f}, "
#         f"Tie = {result['tie_param']:.4f}"
#     )
for result in results:
    probabilities = result["probabilities"]
# print(true_probabilities)
# max_confidence = probabilities.max(dim=1).values
# print(probabilities)
    mean_max_probability = probabilities.max(dim=1).values.mean()
    print(mean_max_probability)

tensor(0.4044)
tensor(0.4525)
tensor(0.4772)
tensor(0.5509)


In [33]:
print(
    train_df["label"]
    .value_counts(normalize=True)
    .sort_index()
)

label
0    0.350333
1    0.340778
2    0.308889
Name: proportion, dtype: float64


In [34]:
print(
    val_df["label"]
    .value_counts(normalize=True)
    .sort_index()
)

label
0    0.350
1    0.341
2    0.309
Name: proportion, dtype: float64


In [35]:
def evaluate_checkpoint_2(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    reward_model = RewardModel(backbone)
    reward_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    reward_model.to(device)
    reward_model.eval()

    total_loss = 0.0
    num_examples = 0
    correct = 0

    # NEW
    all_confidence = []
    all_correct = []

    with torch.no_grad():

        for enc_a, enc_b, labels in val_loader:

            enc_a = {
                k: v.to(device)
                for k, v in enc_a.items()
            }

            enc_b = {
                k: v.to(device)
                for k, v in enc_b.items()
            }

            labels = labels.to(device)

            reward_a = reward_model(**enc_a)
            reward_b = reward_model(**enc_b)

            tie_param = reward_model.get_tie_param()

            prob_a, prob_b, prob_tie = preference_probabilities(
                reward_a,
                reward_b,
                tie_param=tie_param
            )

            probabilities = torch.stack(
                [prob_a, prob_b, prob_tie],
                dim=1
            )

            # -------------------------
            # Prediction
            # -------------------------

            predictions = probabilities.argmax(dim=1)

            correct_mask = predictions == labels

            # -------------------------
            # Confidence
            # -------------------------

            confidence = probabilities.max(dim=1).values

            all_confidence.append(
                confidence.cpu()
            )

            all_correct.append(
                correct_mask.cpu()
            )

            # -------------------------
            # Loss
            # -------------------------

            true_probabilities = probabilities[
                torch.arange(
                    labels.size(0),
                    device=labels.device
                ),
                labels
            ]

            loss = -torch.log(
                true_probabilities
            ).mean()

            total_loss += loss.item() * labels.size(0)
            num_examples += labels.size(0)

            correct += correct_mask.sum().item()

    # Combine all batches
    all_confidence = torch.cat(all_confidence)
    all_correct = torch.cat(all_correct)

    # -------------------------
    # Confidence statistics
    # -------------------------

    correct_confidence = all_confidence[
        all_correct
    ].mean()

    wrong_confidence = all_confidence[
        ~all_correct
    ].mean()

    accuracy = correct / num_examples
    val_loss = total_loss / num_examples

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(
        f"Correct confidence: "
        f"{correct_confidence.item():.4f}"
    )
    print(
        f"Wrong confidence: "
        f"{wrong_confidence.item():.4f}"
    )

    return {
        "val_loss": val_loss,
        "accuracy": accuracy,
        "correct_confidence": correct_confidence.item(),
        "wrong_confidence": wrong_confidence.item(),
        "tie_param": reward_model.get_tie_param().item()
    }
results = []
for i in range(3):
    result = evaluate_checkpoint_2(f"../models/epoch_{i + 1}.pt")
    results.append(result)
    # print(
    #         f"Epoch {result['epoch']}: "
    #         f"Train Loss = {result['train_loss']:.4f}, "
    #         f"Val Loss = {result['val_loss']:.4f}, "
    #         f"Val Accuracy = {result['val_accuracy']:.4f}, "
    #         f"Tie = {result['tie_param']:.4f}"
            
    #     )

Accuracy: 0.4110
Val Loss: 1.0505
Correct confidence: 0.5023
Wrong confidence: 0.4766
Accuracy: 0.5750
Val Loss: 0.8730
Correct confidence: 0.5448
Wrong confidence: 0.4722
Accuracy: 0.8080
Val Loss: 0.5739
Correct confidence: 0.6670
Wrong confidence: 0.5204
